## Welcome to the ebook2audiobook Google Colab!
## Features
- 🔧 **TTS Engines supported**: XTTSv2, Bark, Fairseq, VITS, Tacotron2, Tortoise, GlowTTS, YourTTS- 📚 **Convert multiple file formats**: .epub, .mobi, .azw3, .fb2, .lrf, .rb, .snb, .tcr, .pdf, .txt, .rtf, .doc, .docx, .html, .odt, .azw, .tiff, .tif, .png, .jpg, .jpeg, .bmp
- 🔍 **OCR scanning** for files with text pages as images
- 🔊 **High-quality text-to-speech** from near realtime to near real voice
- 🗣️ **Optional voice cloning** using your own voice file
- 🌐 **Supports 1158 languages** ([supported languages list](https://dl.fbaipublicfiles.com/mms/tts/all-tts-languages.html))
- 💻 **Low-resource friendly** — runs on **2 GB RAM / 1 GB VRAM (minimum)**
- 🎵 **Audiobook output formats**: mono or stereo aac, flac, mp3, m4b, m4a, mp4, mov, ogg, wav, webm
- 🧠 **SML tags supported** — fine-grained control of breaks, pauses, voice switching and more ([see below](#sml-tags-available))
- 🧩 **Optional custom model** using your own trained model (XTTSv2 only, other on request)
- 🎛️ **Fine-tuned preset models** trained by the E2A Team<br/>
     <i>(Contact us if you need additional fine-tuned models, or if you'd like to share yours to the official preset list)</i>
## Want to run locally for free? ⬇
## [Check out the ebook2audiobook github!](https://github.com/pcg1974/ebook2audiobook)

In [ ]:
# @title 📁 Block 1: Mount Google Drive & Set Paths
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/Colab/ebook2audiobook"
TAR_FILE_NAME = "python_env_backup.tar.gz"
DRIVE_TAR_PATH = os.path.join(DRIVE_BACKUP_DIR, TAR_FILE_NAME)

os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print("✅ Google Drive mounted and backup directory initialized.")

In [ ]:
# @title 🛠️ Block 2: Full Setup & Create Drive Snapshot (Slow Setup with Benchmarking)

import os
import subprocess
import time
import shutil

CHECK_MARK = "✅"
CROSS_MARK = "❌"

SCRIPT_DIR = "/content/ebook2audiobook"
VENV_DIR = f"{SCRIPT_DIR}/python_env"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
LOCAL_TAR_PATH = f"/content/{TAR_FILE_NAME}"

os.makedirs(f"{SCRIPT_DIR}/tmp", exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)

# Track timing metrics
timings = {}
overall_start = time.time()

def format_duration(seconds):
    mins, secs = divmod(int(seconds), 60)
    return f"{mins}m {secs}s" if mins > 0 else f"{seconds:.1f}s"

def run_cmd(cmd, desc, cwd=None):
    print(f"\n➜ Starting: {desc}...")
    step_start = time.time()
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd or SCRIPT_DIR)
    elapsed = time.time() - step_start
    
    if res.returncode == 0:
        timings[desc] = elapsed
        print(f"{CHECK_MARK} Done in {format_duration(elapsed)}.")
        return True
    else:
        print(f"{CROSS_MARK} Failed: {desc} (after {format_duration(elapsed)})\nError: {res.stderr.strip()}")
        return False

print("🚀 Starting Full Environment Build...")

# 1. OS Packages & Calibre
os_cmd = (
    "apt-get update -qq && "
    "apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils libxcb-cursor0 libegl1 libopengl0 ffmpeg mediainfo nodejs espeak-ng sox tesseract-ocr tesseract-ocr-eng mecab libmecab-dev mecab-ipadic-utf8 && "
    "touch /etc/mecabrc && "
    "wget -nv -O- https://download.calibre-ebook.com/linux-installer.sh | sudo sh /dev/stdin"
)
run_cmd(os_cmd, "Installing OS packages & Calibre", cwd="/content")

# 2. Git Clone
run_cmd(f"git clone --depth=1 https://github.com/pcg1974/ebook2audiobook.git {SCRIPT_DIR}", "Cloning repository", cwd="/content")

# 3. Python Virtual Environment Setup
venv_build_cmd = (
    f"python3.10 -m venv --without-pip {VENV_DIR} && "
    f"curl -sSL https://bootstrap.pypa.io/get-pip.py | {VENV_PYTHON} && "
    f"{VENV_PYTHON} -m pip install -q --upgrade pip setuptools wheel packaging && "
    f"{VENV_PYTHON} -m pip install -q --upgrade llvmlite numba --only-binary=:all: && "
    f"sed -i '/ext\\/py\\/demucs/d' {SCRIPT_DIR}/requirements.txt && "
    f"{VENV_PYTHON} -m pip install -q --no-cache-dir -r {SCRIPT_DIR}/requirements.txt && "
    f"{VENV_PYTHON} -m pip install -e {SCRIPT_DIR}/ext/py/demucs --no-deps -q && "
    f"{VENV_PYTHON} -m pip install -q unidic-lite && "
    f"{VENV_PYTHON} -m pip uninstall -y unidic"
)
run_cmd(venv_build_cmd, "Building venv & Python dependencies")

# 4. Patch Source Code Bugs
step_start = time.time()
patch_file = f"{SCRIPT_DIR}/lib/classes/device_installer.py"
if os.path.exists(patch_file):
    with open(patch_file, "r", encoding="utf-8") as f:
        code = f.read()
    code = code.replace("tag_ver = _normalize_version(ver_str)", "tag_ver = __import__('packaging.version').version.parse(str(ver_str))")
    code = code.replace("v <= version", "v <= __import__('packaging.version').version.parse(str(version))")
    with open(patch_file, "w", encoding="utf-8") as f:
        f.write(code)
    elapsed = time.time() - step_start
    timings["Patching source code bugs"] = elapsed
    print(f"{CHECK_MARK} Patched source code bugs in {format_duration(elapsed)}.")

# 5. Create Local Snapshot
run_cmd(f"tar -czf {LOCAL_TAR_PATH} -C {SCRIPT_DIR} python_env", "Compressing python_env locally", cwd="/content")

# 6. Sync Snapshot to Google Drive
step_start = time.time()
print("\n➜ Uploading snapshot to Google Drive...")
shutil.copy2(LOCAL_TAR_PATH, DRIVE_TAR_PATH)
elapsed = time.time() - step_start
timings["Uploading snapshot to Google Drive"] = elapsed
print(f"{CHECK_MARK} Uploaded snapshot to Google Drive in {format_duration(elapsed)}.")

# 📊 Timing Benchmark Summary
total_time = time.time() - overall_start

print("\n" + "=" * 55)
print("📊 SETUP TIMING BENCHMARK SUMMARY")
print("=" * 55)
for step_name, duration in timings.items():
    print(f" • {step_name:<38}: {format_duration(duration)}")
print("-" * 55)
print(f" TOTAL BUILD & BACKUP TIME             : {format_duration(total_time)}")
print("=" * 55)
print("\n✨ Setup complete! Run Block 4 to launch the app.")

In [ ]:
# @title 🚁 Relaunch app from existing virtual environment
!cd /content/ebook2audiobook && \
 GRADIO_ALLOWED_PATHS='/content/drive/MyDrive/Colab/ebook2audiobook,/content/ebook2audiobook' \
 VIRTUAL_ENV=/content/ebook2audiobook/python_env \
 /content/ebook2audiobook/python_env/bin/python -u app.py --script_mode native --share

In [ ]:
# @title ⚡ Block 3: Fast Startup (Restore Snapshot from Google Drive)

import os
import subprocess
import shutil

CHECK_MARK = "✅"
CROSS_MARK = "❌"

SCRIPT_DIR = "/content/ebook2audiobook"
VENV_DIR = f"{SCRIPT_DIR}/python_env"
LOCAL_TAR_PATH = f"/content/{TAR_FILE_NAME}"

os.makedirs(f"{SCRIPT_DIR}/tmp", exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)

def run_cmd(cmd, desc, cwd=None):
    print(f"➜ {desc}...")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd or "/content")
    if res.returncode == 0:
        print(f"{CHECK_MARK} Done.")
        return True
    else:
        print(f"{CROSS_MARK} Failed: {desc}\nError: {res.stderr.strip()}")
        return False

print("⚡ Starting Fast Environment Restore...")

# 1. OS Dependencies (Fast check)
if shutil.which("calibre") is None:
    os_cmd = (
        "apt-get update -qq && "
        "apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils libxcb-cursor0 libegl1 libopengl0 ffmpeg mediainfo nodejs espeak-ng sox tesseract-ocr tesseract-ocr-eng mecab libmecab-dev mecab-ipadic-utf8 && "
        "touch /etc/mecabrc && "
        "wget -nv -O- https://download.calibre-ebook.com/linux-installer.sh | sudo sh /dev/stdin"
    )
    run_cmd(os_cmd, "Installing OS system packages")
else:
    print(f"{CHECK_MARK} OS dependencies ready.")

# 2. Clone Repository
if not os.path.exists(SCRIPT_DIR):
    run_cmd(f"git clone --depth=1 https://github.com/pcg1974/ebook2audiobook.git {SCRIPT_DIR}", "Cloning repository")

# 3. Restore Snapshot from Drive
if not os.path.exists(DRIVE_TAR_PATH):
    raise FileNotFoundError(f"❌ Backup snapshot not found at {DRIVE_TAR_PATH}. Run Block 2 first!")

print("☁️ Copying snapshot from Google Drive...")
shutil.copy2(DRIVE_TAR_PATH, LOCAL_TAR_PATH)

run_cmd(f"tar -xzf {LOCAL_TAR_PATH} -C {SCRIPT_DIR}", "Extracting python_env locally")

# 4. Patch device_installer.py
patch_file = f"{SCRIPT_DIR}/lib/classes/device_installer.py"
if os.path.exists(patch_file):
    with open(patch_file, "r", encoding="utf-8") as f:
        code = f.read()
    code = code.replace("tag_ver = _normalize_version(ver_str)", "tag_ver = __import__('packaging.version').version.parse(str(ver_str))")
    code = code.replace("v <= version", "v <= __import__('packaging.version').version.parse(str(version))")
    with open(patch_file, "w", encoding="utf-8") as f:
        f.write(code)

print(f"\n{CHECK_MARK} Environment restored successfully! Run Block 4 to launch the app.")

In [ ]:
# @title 🚀 Block 4: Launch ebook2audiobook

import os

SCRIPT_DIR = "/content/ebook2audiobook"
VENV_DIR = f"{SCRIPT_DIR}/python_env"
VENV_PYTHON = f"{VENV_DIR}/bin/python"

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["TTS_CACHE"] = f"{SCRIPT_DIR}/models"
os.environ["TESSDATA_PREFIX"] = f"{SCRIPT_DIR}/models/tessdata"
os.environ["TMPDIR"] = "/tmp"
os.environ["SCRIPT_MODE"] = "native"
os.environ["MPLBACKEND"] = "Agg"
os.environ["GRADIO_ALLOWED_PATHS"] = f"{DRIVE_BACKUP_DIR},{SCRIPT_DIR}"

for d in ["models", "models/tessdata", "tmp", "run", "audiobooks", "ebooks", "voices"]:
    os.makedirs(f"{SCRIPT_DIR}/{d}", exist_ok=True)

print("🚀 Launching ebook2audiobook Gradio App...")

try:
    get_ipython().system(
        f"cd {SCRIPT_DIR} && "
        f"VIRTUAL_ENV={VENV_DIR} {VENV_PYTHON} -u app.py --script_mode native --share"
    )
except Exception as e:
    print(f"❌ Error starting app.py: {e}")